[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/08_end_to_end_ml_system/project.ipynb)

# Project 08 — Hệ thống phát hiện gian lận end-to-end

> Một ví điện tử xử lý hàng chục nghìn giao dịch thẻ mỗi ngày; ~1% là gian lận. Giao dịch gian lận được duyệt → mất **toàn bộ số tiền**. Mỗi giao dịch chuyển sang **kiểm duyệt thủ công** tốn **15.000đ** (nhân sự + khách phải chờ). Bạn xây **toàn bộ hệ thống**: mô hình, chính sách quyết định theo chi phí, gói dịch vụ có kiểm thử, API, giám sát — như ngoài thực tế.

**Bạn sẽ làm** (notebook này + thư mục mã nguồn):
1. Khung bài toán và EDA; vì sao accuracy vô nghĩa.
2. So sánh mô hình bằng PR-AUC **và** calibration — chọn mô hình phục vụ được quyết định theo xác suất.
3. Chính sách **tổn thất kỳ vọng**: kiểm duyệt khi $p \times \text{số tiền} > 15.000$đ; so với các baseline bằng **tiền**.
4. Đóng gói thành dịch vụ: `fraud_service/` (package `fraud`, CLI huấn luyện có cổng chất lượng, API FastAPI, giám sát PSI, bộ test, Dockerfile, Makefile) — chạy huấn luyện và test như CI.
5. Gọi API; mô phỏng một tuần dữ liệu trôi và chạy báo cáo giám sát; kế hoạch triển khai.

**Tiêu chí đạt**: PR-AUC validation của mô hình chọn > 0.06 và xác suất TB lệch < 50% so với tỉ lệ gian lận · chi phí chính sách theo mô hình thấp hơn mọi baseline ít nhất 40% so với "duyệt tất cả" · CLI huấn luyện và toàn bộ test của dịch vụ đạt · báo cáo giám sát phát hiện trôi số tiền.

**Thư mục mã nguồn**: `fraud_service/` là **bản tham chiếu hoàn chỉnh**. Cách học tốt nhất: tự viết dịch vụ của bạn trong `my_service/` theo cùng cấu trúc (xem `fraud_service/README.md`), rồi so sánh.

**Chương liên quan**: 14, 19 (metric & chi phí khi mất cân bằng), 37 (calibration), 39 (đóng gói, metadata, cổng chất lượng), 40 (API, test, Docker, CI, giám sát, triển khai).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/08_end_to_end_ml_system

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

SERVICE = repo_path("projects", "08_end_to_end_ml_system", "fraud_service")
sys.path.insert(0, str(SERVICE))                                        # để import package `fraud` của dịch vụ tham chiếu
from fraud.config import CATEGORICAL, NUMERIC, REVIEW_COST_VND  # noqa: E402
from fraud.data import load_transactions, split  # noqa: E402
from fraud.features import add_features  # noqa: E402

warnings.filterwarnings("ignore", message="Using `httpx` with")
transactions = load_transactions(repo_path("data", "sample", "fraud_transactions.csv"))
train, valid, test = split(transactions)
print(transactions.shape, f"| gian lận: {transactions.is_fraud.mean():.2%} | train {len(train):,} / valid {len(valid):,} / test {len(test):,}")
transactions.head()

## Bước 1 — EDA và cái bẫy accuracy

Tính trên train: `fraud_by_category` (tỉ lệ gian lận theo `merchant_category`), `fraud_rate_foreign` và `fraud_rate_domestic`, `median_amount_fraud` và `median_amount_legit`; `accuracy_all_legit` = accuracy trên valid khi dự đoán **mọi** giao dịch là hợp lệ.

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
fraud_by_category = train.groupby("merchant_category").is_fraud.mean().sort_values(ascending=False)
fraud_rate_foreign = train.is_fraud[train.is_foreign].mean()
fraud_rate_domestic = train.is_fraud[~train.is_foreign].mean()
median_amount_fraud = train.amount_vnd[train.is_fraud == 1].median()
median_amount_legit = train.amount_vnd[train.is_fraud == 0].median()
accuracy_all_legit = (valid.is_fraud == 0).mean()
### END CODE HERE ###
print(fraud_by_category.round(4).to_dict())
print(f"nước ngoài {fraud_rate_foreign:.2%} vs trong nước {fraud_rate_domestic:.2%} | trung vị số tiền: gian lận {median_amount_fraud:,.0f}đ vs hợp lệ {median_amount_legit:,.0f}đ")
print(f"accuracy khi 'mọi giao dịch hợp lệ': {accuracy_all_legit:.2%} — và bỏ lọt 100% gian lận")

In [ ]:
check_greater(fraud_rate_foreign, 3 * fraud_rate_domestic, "giao dịch nước ngoài rủi ro hơn nhiều")
check_greater(median_amount_fraud, 1.5 * median_amount_legit, "giao dịch gian lận thường lớn hơn")
check_greater(accuracy_all_legit, 0.98, "accuracy cao mà vô dụng")

## Bước 2 — So sánh mô hình: xếp hạng **và** calibration

Ba ứng viên (pipeline chung: `FunctionTransformer(add_features)` → chuẩn hoá `NUMERIC`, one-hot `CATEGORICAL`):
`logistic` (`LogisticRegression(max_iter=2000)`), `logistic_balanced` (thêm `class_weight="balanced"`), `boosting` (`HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=50, random_state=0)`). Huấn luyện trên train, đánh giá trên valid → `comparison` (DataFrame index tên, cột `pr_auc`, `roc_auc`, `brier`, `mean_pred`). Lưu xác suất valid vào dict `valid_proba`.

In [ ]:
def make_pipeline_for(estimator):
    columns = ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(FunctionTransformer(add_features), columns, estimator)


candidates = {
    "logistic": make_pipeline_for(LogisticRegression(max_iter=2000)),
    "logistic_balanced": make_pipeline_for(LogisticRegression(max_iter=2000, class_weight="balanced")),
    "boosting": make_pipeline_for(HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=50, random_state=0)),
}
### START CODE HERE ### (≈ 7 dòng)
# STARTER: valid_proba, comparison = {}, None
valid_proba, rows = {}, {}
for name, model in candidates.items():
    model.fit(train, train.is_fraud)
    p = valid_proba[name] = model.predict_proba(valid)[:, 1]
    rows[name] = {"pr_auc": average_precision_score(valid.is_fraud, p), "roc_auc": roc_auc_score(valid.is_fraud, p),
                  "brier": brier_score_loss(valid.is_fraud, p), "mean_pred": p.mean()}
comparison = pd.DataFrame(rows).T
### END CODE HERE ###
print(comparison.round(4), f"\n(tỉ lệ gian lận thật trên valid: {valid.is_fraud.mean():.4f}; PR-AUC của mô hình ngẫu nhiên ≈ tỉ lệ này)")

In [ ]:
check_equal(sorted(comparison.index), ["boosting", "logistic", "logistic_balanced"], "ba ứng viên")
check_greater(comparison.loc["logistic", "pr_auc"], 0.06, "PR-AUC ≫ tỉ lệ nền 1%")
check_less(abs(comparison.loc["logistic", "mean_pred"] / valid.is_fraud.mean() - 1), 0.5, "logistic hiệu chỉnh tốt")
check_greater(comparison.loc["logistic_balanced", "mean_pred"], 10 * valid.is_fraud.mean(), "class_weight='balanced' thổi phồng xác suất")

- PR-AUC thấp (~0.1) là **bình thường** với gian lận 1% và vài đặc trưng — vẫn gấp ~10 lần ngẫu nhiên. Đừng so PR-AUC giữa các bài toán có tỉ lệ dương khác nhau.
- `class_weight="balanced"` giữ nguyên **thứ hạng** nhưng làm xác suất trung bình tăng vọt (mô hình "tưởng" 50% là gian lận) — vô dụng cho quyết định cần **xác suất thật** (chương 19, 37).
- Boosting không tốt hơn trên dữ liệu ít ca dương (~150 ca gian lận trong train) và quan hệ gần tuyến tính. Chọn **logistic**.

## Bước 3 — Chính sách tổn thất kỳ vọng

Duyệt thẳng giao dịch có xác suất gian lận $p$ và số tiền $A$ → tổn thất kỳ vọng $pA$; kiểm duyệt tốn $c = 15.000$đ (và chặn được gian lận). Kiểm duyệt khi $pA > c$ — ngưỡng **phụ thuộc số tiền**: giao dịch 100.000đ chỉ kiểm duyệt khi $p > 15\%$, giao dịch 10 triệu khi $p > 0.15\%$.

Viết `decide(p, amount)` và `total_cost(df, review)` (chi phí thực tế: kiểm duyệt → $c$; duyệt thẳng giao dịch gian lận → mất số tiền). Tính chi phí trên valid (triệu đồng) → `policy_costs` (Series) cho: `"duyệt tất cả"`, `"kiểm duyệt tất cả"`, `"luật: nước ngoài hoặc (đêm và > 1tr)"`, `"logistic: p > 0.5"`, `"logistic: tổn thất kỳ vọng"`, `"logistic_balanced: tổn thất kỳ vọng"`.

In [ ]:
def decide(p, amount):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return np.asarray(p) > 0.5
    return np.asarray(p, float) * np.asarray(amount, float) > REVIEW_COST_VND
    ### END CODE HERE ###


def total_cost(df, review):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return 0.0
    review = np.asarray(review, bool)
    fraud = df.is_fraud.to_numpy() == 1
    return float(np.where(review, REVIEW_COST_VND, np.where(fraud, df.amount_vnd.to_numpy(), 0.0)).sum())
    ### END CODE HERE ###


feats_valid = add_features(valid)
### START CODE HERE ### (≈ 8 dòng)
# STARTER: policies = {}        # tên chính sách → mảng bool "kiểm duyệt" trên valid
# STARTER: policy_costs = None
policies = {
    "duyệt tất cả": np.zeros(len(valid), bool),
    "kiểm duyệt tất cả": np.ones(len(valid), bool),
    "luật: nước ngoài hoặc (đêm và > 1tr)": (feats_valid.is_foreign | ((feats_valid.is_night == 1) & (feats_valid.amount_vnd > 1e6))).to_numpy(),
    "logistic: p > 0.5": valid_proba["logistic"] > 0.5,
    "logistic: tổn thất kỳ vọng": decide(valid_proba["logistic"], valid.amount_vnd),
    "logistic_balanced: tổn thất kỳ vọng": decide(valid_proba["logistic_balanced"], valid.amount_vnd),
}
policy_costs = pd.Series({name: total_cost(valid, review) / 1e6 for name, review in policies.items()})
### END CODE HERE ###
review_rates = pd.Series({name: np.mean(r) for name, r in policies.items()})
print(pd.DataFrame({"chi phí (triệu đồng)": policy_costs, "tỉ lệ kiểm duyệt": review_rates}).round(3).sort_values("chi phí (triệu đồng)"))

In [ ]:
check_equal(decide([0.2, 0.001, 0.01], [1e5, 1e7, 1e6]).tolist(), [True, False, False], "ngưỡng phụ thuộc số tiền")
check_close(total_cost(valid.head(3).assign(is_fraud=[1, 0, 1], amount_vnd=[1e6, 2e6, 3e6]), [True, False, False]), 15_000 + 3e6, message="công thức chi phí")
best = policy_costs["logistic: tổn thất kỳ vọng"]
check_true(best == policy_costs.min(), "chính sách tổn thất kỳ vọng (mô hình hiệu chỉnh) rẻ nhất")
check_less(best, 0.6 * policy_costs["duyệt tất cả"], "rẻ hơn 'duyệt tất cả' ít nhất 40%")
check_greater(policy_costs["logistic_balanced: tổn thất kỳ vọng"], best, "mô hình mất hiệu chỉnh → chính sách tốn hơn")

**Nhận xét:** Cùng một mô hình logistic, ngưỡng cố định 0.5 gần như không kiểm duyệt gì (xác suất hiếm khi vượt 0.5 khi tỉ lệ nền 1%) — chi phí ngang "duyệt tất cả". Chính sách tổn thất kỳ vọng tập trung kiểm duyệt vào giao dịch **lớn và đáng ngờ**: chỉ ~13% giao dịch nhưng cắt chi phí khoảng 2/3. Mô hình `balanced` có cùng thứ hạng nhưng xác suất bị thổi phồng → kiểm duyệt quá nhiều, tốn hơn. Metric quyết định không phải PR-AUC mà là **tiền**; và tiền phụ thuộc **calibration**.

## Bước 4 — Đóng gói thành dịch vụ

Mã nguồn dịch vụ tham chiếu (đọc từng file — chúng ngắn):

```
fraud_service/
├── fraud/config.py       # chi phí kiểm duyệt, danh sách đặc trưng — một nguồn sự thật
├── fraud/features.py     # add_features (dùng chung train/serve)
├── fraud/data.py         # load, validate (hợp đồng dữ liệu), split
├── fraud/model.py        # build_model, decide, total_cost, baseline_policies
├── fraud/train.py        # CLI: kiểm tra dữ liệu → validation + cổng chất lượng → refit → test → model.joblib + metadata.json
├── fraud/api.py          # FastAPI: /health, /v1/score, /v1/policy (kiểm tra hash khi khởi động)
├── fraud/monitoring.py   # PSI so với mốc huấn luyện lưu trong metadata
├── tests/                # dữ liệu & đặc trưng, mô hình (chất lượng tối thiểu, chính sách, hướng, bất biến), API (parity), giám sát
├── Dockerfile · .dockerignore · requirements.txt · Makefile · pyproject.toml
```

Chạy đúng những lệnh CI sẽ chạy: huấn luyện (vào thư mục tạm) và bộ test.

In [ ]:
model_dir = Path(tempfile.mkdtemp()) / "fraud_model"


def run(args):
    result = subprocess.run([sys.executable, *args], cwd=SERVICE, capture_output=True, text=True, encoding="utf-8",
                            errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print("\n".join((result.stdout + result.stderr).strip().splitlines()[-8:]))
    return result.returncode


train_code = run(["-m", "fraud.train", "--out", str(model_dir)])
test_code = run(["-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"])
metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"\nmã thoát: train {train_code}, pytest {test_code} | metadata: {sorted(metadata)}")

In [ ]:
check_equal(train_code, 0, "huấn luyện qua cổng chất lượng")
check_equal(test_code, 0, "toàn bộ test của dịch vụ đạt")
check_less(metadata["test"]["cost_model_vnd"], 0.6 * metadata["test"]["cost_baselines_vnd"]["duyệt tất cả"], "trên test, chi phí giảm ≥ 40% so với duyệt tất cả")

## Bước 5 — Gọi API

Trong production: `make serve` (uvicorn) hoặc `make docker-build && make docker-run`. Ở đây dùng `TestClient` với mô hình vừa huấn luyện.

In [ ]:
os.environ["FRAUD_MODEL_DIR"] = str(model_dir)
from fastapi.testclient import TestClient  # noqa: E402

from fraud.api import app  # noqa: E402

examples = [
    {"amount_vnd": 120_000, "hour": 14, "merchant_category": "food", "distance_from_home_km": 2.0, "is_foreign": False, "n_tx_last_24h": 1, "card_age_days": 2000},
    {"amount_vnd": 9_500_000, "hour": 2, "merchant_category": "electronics", "distance_from_home_km": 35.0, "is_foreign": True, "n_tx_last_24h": 6, "card_age_days": 40},
    {"amount_vnd": 450_000, "hour": 23, "merchant_category": "online_games", "distance_from_home_km": 8.0, "is_foreign": False, "n_tx_last_24h": 4, "card_age_days": 300},
]
with TestClient(app) as client:
    print(client.get("/health").json(), client.get("/v1/policy").json())
    for ex, scored in zip(examples, client.post("/v1/score", json=examples).json()):
        print(f"{ex['amount_vnd']:>11,.0f}đ {ex['merchant_category']:12s} → p = {scored['fraud_probability']:.4f}, tổn thất kỳ vọng {scored['expected_loss_vnd']:>9,.0f}đ → {scored['decision']}")
    bad = client.post("/v1/score", json=[{**examples[0], "hour": 25}])
    print("đầu vào sai →", bad.status_code)

## Bước 6 — Giám sát một tuần dữ liệu mới

Mô phỏng tuần sau ra mắt một chương trình khuyến mãi **game online**: số giao dịch `online_games` tăng gấp 3 và số tiền trung bình tăng 50%. Tạo `week` (5.000 giao dịch từ test, lấy mẫu có trọng số + nhân số tiền), chạy `drift_report` với mốc trong `metadata` → `report`.

In [ ]:
from fraud.monitoring import drift_report  # noqa: E402

### START CODE HERE ### (≈ 4 dòng)
weights = np.where(test.merchant_category == "online_games", 3.0, 1.0)
week = test.sample(5000, replace=True, weights=weights, random_state=0).assign(amount_vnd=lambda d: d.amount_vnd * 1.5)
report = drift_report(metadata["reference_profile"], week)
### END CODE HERE ###
print(report.to_string(index=False))
print(f"\ntỉ lệ online_games: train {train.merchant_category.eq('online_games').mean():.1%} → tuần mới {week.merchant_category.eq('online_games').mean():.1%}")

In [ ]:
check_equal(len(week), 5000, "5.000 giao dịch")
check_true(report.set_index("feature").loc["log_amount", "status"] in ("WARN", "ALERT"), "phát hiện trôi số tiền")
check_less(report.set_index("feature").loc["card_age_days", "psi"], 0.1, "đặc trưng không đổi thì không báo động")

**Nhận xét:** Trôi số tiền bị phát hiện ngay (PSI cảnh báo). Nhưng báo cáo hiện tại **không** giám sát cơ cấu `merchant_category` — một thiếu sót: thêm PSI cho biến phân loại (chương 40) vào `fraud/monitoring.py`. Hành động khi có cảnh báo: (1) xác nhận nguyên nhân với bộ phận kinh doanh (khuyến mãi — trôi "có chủ đích"); (2) kiểm tra chi phí kiểm duyệt: số tiền tăng → nhiều giao dịch vượt ngưỡng $pA > c$ → tỉ lệ kiểm duyệt tăng, cần báo trước cho đội kiểm duyệt; (3) theo dõi tỉ lệ gian lận khi nhãn về (thường vài tuần — khiếu nại chargeback) trước khi quyết định huấn luyện lại.

## Bước 7 — Kế hoạch triển khai và báo cáo

Viết (1) kế hoạch triển khai 4 bước theo chương 40 (shadow → canary → A/B hoặc so sánh chi phí → toàn bộ), mỗi bước kèm tiêu chí dừng/rollback; (2) báo cáo 6–8 câu cho giám đốc rủi ro: chi phí giảm bao nhiêu so với luật hiện tại, tỉ lệ kiểm duyệt, giả định (chi phí kiểm duyệt cố định, kiểm duyệt bắt 100% gian lận), rủi ro (trôi dữ liệu, kẻ gian thích nghi), giám sát.

✍️ Kế hoạch và báo cáo:

**Kế hoạch mẫu:** (1) **Shadow 2 tuần**: chấm điểm song song luật hiện tại, không ảnh hưởng quyết định; tiêu chí: độ trễ p99 < 50 ms, lỗi < 0.1%, tỉ lệ "review" dự kiến ~13%, phân phối xác suất khớp validation. (2) **Canary 10% giao dịch** (gán theo băm mã thẻ): rollback nếu tỉ lệ kiểm duyệt > 20% hoặc độ trễ vượt ngưỡng. (3) **So sánh chi phí** nhóm mô hình vs nhóm luật trong 4–6 tuần (đủ thời gian để chargeback về), kiểm định chênh lệch chi phí/giao dịch. (4) **Toàn bộ**, giữ 2% giao dịch theo luật cũ làm nhóm đối chứng lâu dài (vòng phản hồi — giao dịch bị kiểm duyệt có nhãn khác giao dịch được duyệt).

**Báo cáo mẫu:** Mô hình logistic với chính sách "kiểm duyệt khi tổn thất kỳ vọng > chi phí kiểm duyệt" giảm tổng chi phí gian lận + kiểm duyệt khoảng 2/3 so với duyệt tất cả và khoảng một nửa so với luật hiện tại trên dữ liệu kiểm tra, trong khi chỉ kiểm duyệt ~13% giao dịch (chủ yếu giao dịch lớn, nước ngoài, ban đêm). Giả định: chi phí kiểm duyệt cố định 15.000đ và kiểm duyệt chặn được mọi gian lận — cần xác nhận với đội vận hành. Rủi ro chính: kẻ gian **thích nghi** (đổi hành vi để né mô hình) và trôi dữ liệu do khuyến mãi; giám sát PSI hằng ngày, tỉ lệ kiểm duyệt, tỉ lệ chargeback hằng tuần, huấn luyện lại hằng tháng qua cổng chất lượng. Hệ thống đã có kiểm thử tự động (dữ liệu, mô hình, API, giám sát), đóng gói Docker và có thể rollback bằng phiên bản mô hình trước.